In [1]:
# !pip install gradio ollama

### Block 1: Libraries

In [2]:
# --- Block 1: Libraries ---
import gradio as gr
import ollama

### Block 2: Process (The Core Engine)

In [3]:
def negotiate(message, history, model_choice, contract_snippet, relationship, tone, prop_org, prop_persona, opp_org, opp_persona, opp_interests, opp_leverage, opp_batna, opp_walkaway):
    
    # Helper function to clean Gradio's complex text format
    def extract_text(content):
        if isinstance(content, str):
            return content
        elif isinstance(content, list):
            return " ".join([item.get("text", "") for item in content if isinstance(item, dict)])
        return str(content)

    system_prompt = f"""You are the Opponent in a negotiation simulation. 
    Relationship: {relationship} | Tone: {tone}
    
    SPECIFIC CLAUSES UNDER NEGOTIATION:
    {contract_snippet}
    
    PERSONAS:
    - Opponent (You): {opp_persona} ({opp_org})
    - Proponent (User): {prop_persona} ({prop_org})
    
    YOUR STRATEGY & CONSTRAINTS:
    - Interests: {opp_interests}
    - Leverage: {opp_leverage}
    - BATNA: {opp_batna}
    - WALK-AWAY: {opp_walkaway}
    
    RULES: Stay in character. Be concise. Defend your walk-away conditions aggressively. Do not be easily persuaded."""

    llm_messages = [{"role": "system", "content": system_prompt}]
    
    for chat_turn in history:
        llm_messages.append({
            "role": chat_turn["role"], 
            "content": extract_text(chat_turn["content"])
        })
        
    llm_messages.append({"role": "user", "content": extract_text(message)})

    try:
        response = ollama.chat(model=model_choice, messages=llm_messages)
        return response['message']['content']
    except Exception as e:
        return f"Ensure 'ollama run {model_choice}' was successful. Error: {e}"

### Block 3: I/O

In [4]:
# --- BLOCK 3: THE INTERFACE (Gradio 6.0) ---
with gr.Blocks() as demo:
    gr.Markdown("# ⚖️ The Standoff: A Negotiation Simulation")
    
    with gr.Row():
        with gr.Column(scale=1):
            with gr.Accordion("⚙️ Engine & Contract", open=True):
                model_input = gr.Textbox(value="llama3.1:8b", label="Local Model Name")
                snippet_input = gr.Textbox(
                    label="Specific Clauses to be discussed", 
                    value="Paste contract snippet here (focused snippet will allow for faster load times)", 
                    lines=5
                )
                
            with gr.Accordion("🎭 Personas", open=False):
                rel_input = gr.Textbox(label="Relationship", value="BPO Provider vs Client")
                tone_input = gr.Textbox(label="Tone", value="Clinical and firm")
                opp_org_input = gr.Textbox(label="Opponent Org", value="BPO Corp")
                opp_pers_input = gr.Textbox(label="Opponent Persona", value="Harvard MBA, 5yr experience, risk-averse")
                prop_org_input = gr.Textbox(label="Your Org", value="Tech Startup")
                prop_pers_input = gr.Textbox(label="Your Persona", value="Experienced Entrepreneur")

            with gr.Accordion("🛡️ Opponent's Constraints", open=False):
                int_input = gr.Textbox(label="Interests", value="Win the contract with minimum revision", lines=3)
                lev_input = gr.Textbox(label="Leverage", value="User needs to launch next month")
                batna_input = gr.Textbox(label="BATNA", value="Lower margin for less demanding client")
                walk_input = gr.Textbox(label="Walk-Away Conditions", value="Allowing client to go direct")

        with gr.Column(scale=2):
            gr.ChatInterface(fn=negotiate, additional_inputs=[
                model_input, snippet_input, rel_input, tone_input,
                prop_org_input, prop_pers_input, opp_org_input, opp_pers_input,
                int_input, lev_input, batna_input, walk_input
            ])

# Launch in a new browser tab instead of inside the notebook cell
demo.launch(inbrowser=True, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
